In [1]:
import sys
from pathlib import Path

# Make sure project root is on sys.path
PROJECT_ROOT = Path.cwd().parent  # notebooks/ -> project root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd

from src.config import (
    DATA_INTERIM_DIR,
    DATA_PROCESSED_DIR,
    GEO_LIST_36,
)

PROJECT_ROOT, DATA_INTERIM_DIR, DATA_PROCESSED_DIR, len(GEO_LIST_36)


(PosixPath('/Users/princebhanusteta/Documents/Projects/euro-macro-inflation-income'),
 PosixPath('/Users/princebhanusteta/Documents/Projects/euro-macro-inflation-income/data_interim'),
 PosixPath('/Users/princebhanusteta/Documents/Projects/euro-macro-inflation-income/data_processed'),
 36)

In [2]:
hicp_index_q = pd.read_parquet(DATA_INTERIM_DIR / "hicp_index_quarterly_long.parquet")
hicp_inflation_q = pd.read_parquet(DATA_INTERIM_DIR / "hicp_inflation_quarterly_long.parquet")
income_q = pd.read_parquet(DATA_INTERIM_DIR / "income_quarterly_long.parquet")
employment_q = pd.read_parquet(DATA_INTERIM_DIR / "employment_quarterly_long.parquet")

print("HICP index:", hicp_index_q.shape)
print("HICP inflation:", hicp_inflation_q.shape)
print("Income quarterly:", income_q.shape)
print("Employment quarterly:", employment_q.shape)

hicp_index_q.head()


HICP index: (46800, 7)
HICP inflation: (46800, 7)
Income quarterly: (193536, 9)
Employment quarterly: (43200, 9)


,freq,unit,coicop,geo,timeQuarter,hicpIndex,nMonths
0,M,I15,CP00,AT,2000Q1,74.683333,3
1,M,I15,CP00,AT,2000Q2,74.83,3
2,M,I15,CP00,AT,2000Q3,74.923333,3
3,M,I15,CP00,AT,2000Q4,75.64,3
4,M,I15,CP00,AT,2001Q1,76.166667,3


In [3]:
# Canonical quarterly range for the whole project
quarter_index = pd.period_range("2000Q1", "2024Q4", freq="Q-DEC")
len(quarter_index), quarter_index[0], quarter_index[-1]


(100, Period('2000Q1', 'Q-DEC'), Period('2024Q4', 'Q-DEC'))

In [4]:
# Make sure we only keep our 36 geos (should already be true, this is just defensive)
income_q = income_q[income_q["geo"].isin(GEO_LIST_36)].copy()

income_dims = ["geo", "quantile", "indic_il", "currency"]

# Set a MultiIndex using dims + timeQuarter
income_q_idxed = (
    income_q
    .set_index(income_dims + ["timeQuarter"])
    .sort_index()
)

# Build full MultiIndex = all geos × all quantiles × both indicators × currencies × all quarters
full_index = pd.MultiIndex.from_product(
    [
        GEO_LIST_36,
        income_q["quantile"].unique(),
        income_q["indic_il"].unique(),
        income_q["currency"].unique(),
        quarter_index,
    ],
    names=income_dims + ["timeQuarter"],
)

income_q_full = income_q_idxed.reindex(full_index)

# Bring dims back as columns
income_q_full = income_q_full.reset_index()

income_q_full.head()


,geo,quantile,indic_il,currency,timeQuarter,freq,timeStr,incomeValue,timeYear
0,AT,D1,SHARE,EUR,2000Q1,A,2000,4.0,2000
1,AT,D1,SHARE,EUR,2000Q2,A,2000,4.0,2000
2,AT,D1,SHARE,EUR,2000Q3,A,2000,4.0,2000
3,AT,D1,SHARE,EUR,2000Q4,A,2000,4.0,2000
4,AT,D1,SHARE,EUR,2001Q1,A,2001,4.0,2001


In [5]:
at_d1_share = income_q_full[
    (income_q_full["geo"] == "AT")
    & (income_q_full["quantile"] == "D1")
    & (income_q_full["indic_il"] == "SHARE")
]

at_d1_share[at_d1_share["timeQuarter"].dt.year == 2002].head()


,geo,quantile,indic_il,currency,timeQuarter,freq,timeStr,incomeValue,timeYear
8,AT,D1,SHARE,EUR,2002Q1,NaN,NaN,<NA>,NaT
9,AT,D1,SHARE,EUR,2002Q2,NaN,NaN,<NA>,NaT
10,AT,D1,SHARE,EUR,2002Q3,NaN,NaN,<NA>,NaT
11,AT,D1,SHARE,EUR,2002Q4,NaN,NaN,<NA>,NaT


In [6]:
# Drop timeStr + timeYear from income, timeStr from employment
income_q_full = income_q_full.drop(columns=["timeStr", "timeYear"])
employment_q = employment_q.drop(columns=["timeStr"])

print(income_q_full.dtypes) 
print("\n")
print(employment_q.dtypes)

geo                   object
quantile              object
indic_il              object
currency              object
timeQuarter    period[Q-DEC]
freq                  object
incomeValue          Float64
dtype: object


freq                      object
unit                      object
nace_r2                   object
s_adj                     object
na_item                   object
geo                       object
employmentIndex          Float64
timeQuarter        period[Q-DEC]
dtype: object


In [7]:
# Filter HICP to all-items HICP (CP00)
hicp_index_cp00 = hicp_index_q[hicp_index_q["coicop"] == "CP00"].copy()
hicp_inflation_cp00 = hicp_inflation_q[hicp_inflation_q["coicop"] == "CP00"].copy()

print("HICP index CP00:", hicp_index_cp00.shape)
print("HICP inflation CP00:", hicp_inflation_cp00.shape)

# Check uniqueness per geo-quarter
idx_counts = (
    hicp_index_cp00
    .groupby(["geo", "timeQuarter"])["hicpIndex"]
    .size()
)
inf_counts = (
    hicp_inflation_cp00
    .groupby(["geo", "timeQuarter"])["hicpInflation"]
    .size()
)
emp_counts = (
    employment_q
    .groupby(["geo", "timeQuarter"])["employmentIndex"]
    .size()
)

print("\nAny HICP index CP00 duplicates per geo-quarter?", (idx_counts != 1).any())
print("Any HICP inflation CP00 duplicates per geo-quarter?", (inf_counts != 1).any())
print("Any employment duplicates per geo-quarter?", (emp_counts != 1).any())


HICP index CP00: (3600, 7)
HICP inflation CP00: (3600, 7)

Any HICP index CP00 duplicates per geo-quarter? False
Any HICP inflation CP00 duplicates per geo-quarter? False
Any employment duplicates per geo-quarter? True


### Choosing a single headline employment series

The raw Eurostat employment dataset contains **multiple series per country–quarter**.  
Besides `geo` and `timeQuarter`, the data also vary by:

- `unit` (e.g. I15_HW: index, 2015=100, hours worked),
- `nace_r2` (economic activity coverage),
- `s_adj` (seasonal / calendar adjustment),
- `na_item` (employment concept).

When I group only by `(geo, timeQuarter)` I get duplicates, which is expected: there are
several valid employment definitions per country–quarter.

For the **master macro panel** I want a single, consistent “headline” employment index
per country. Following the project guide, I choose:

- `freq = "Q"` (quarterly),
- `unit = "I15_HW"` (index, 2015=100, hours worked),
- `nace_r2 = "A"` (total economy),
- `s_adj = "SCA"` (seasonally and calendar adjusted),
- `na_item = "EMP_DC"` (employment, domestic concept).

I filter to this combination and then check that there is exactly **one row per
(geo, timeQuarter)**. This filtered series is what will be merged into the master
panel; the other variants remain available in the raw/interim data but are not used
for the core analysis.


In [8]:
# Inspect how many distinct values we have (just to confirm)
print(employment_q[["freq", "unit", "nace_r2", "s_adj", "na_item"]].nunique())

# Define the headline filter we want
EMP_FILTER = {
    "freq": "Q",
    "unit": "I15_HW",        # index 2015=100, hours worked
    "nace_r2": "TOTAL",      # Total of all sectors
    "s_adj": "SCA",          # seasonally & calendar adjusted
    "na_item": "EMP_DC",     # domestic concept employment
}


emp_headline = employment_q.copy()
for col, val in EMP_FILTER.items():
    emp_headline = emp_headline[emp_headline[col] == val]

print("Headline employment shape:", emp_headline.shape)

# Check again for duplicates per geo-quarter
emp_counts = (
    emp_headline
    .groupby(["geo", "timeQuarter"])["employmentIndex"]
    .size()
)

print("Any employment duplicates per geo-quarter after filtering?",
      (emp_counts != 1).any())


freq        1
unit        1
nace_r2    12
s_adj       1
na_item     1
dtype: int64
Headline employment shape: (3600, 8)
Any employment duplicates per geo-quarter after filtering? False


In [9]:
# Keep a copy with metadata just in case
emp_headline_full = emp_headline.copy()

# Slim version for merging into the master panel
emp_headline = emp_headline[["geo", "timeQuarter", "employmentIndex"]].copy()


### Design of the master datasets

To keep the project both **clean** and **flexible**, I build *three* related panels from
the quarterly data.

---

### 1. Macro panel (headline HICP only)

The first dataset is the **macro master panel**, focused on the core story:

> How do **headline inflation**, **household income** and **employment** evolve across
> European countries and over time?

Structure:

- One row per:  
  **(geo, quantile, income indicator, currency, timeQuarter)**
- Columns:
  - `incomeValue` (quarterly version of the income quantiles)
  - `hicpIndex` (HICP **CP00**, all-items index)
  - `hicpIndex_nMonths` (how many monthly index observations exist in the quarter)
  - `hicpInflation` (HICP **CP00**, y/y inflation rate)
  - `hicpInflation_nMonths` (how many monthly inflation observations exist in the quarter)
  - `hicp_nMonths_min` (combined quality flag: min of index and inflation month-counts)
  - `employmentIndex` (headline employment index, Q, I15_HW, TOTAL, SCA, EMP_DC)

Key choices:

- For inflation I use **CP00 only** (all-items HICP), which is the standard
  “headline inflation” used in macro analysis and policy.
- Income is expanded from annual to quarterly, including explicit `NaN` values for
  2002 (which is missing in the original Eurostat source).
- Employment is restricted to a **single, well-defined** series per country
  (total economy, seasonally and calendar adjusted, hours worked index).
- Month-coverage flags are kept so later analyses can filter or downweight
  quarters with incomplete monthly data.

This panel is saved as:

- `data_processed/master_quarterly_panel.parquet`

and is the main dataset for EDA, modelling, and dashboarding.

---

### 2. Category panel (HICP by COICOP)

The second dataset is a **category-level panel** that keeps the full HICP
breakdown by COICOP:

> How do different components of inflation (food, energy, housing, etc.) behave
> across countries and over time?

Structure:

- One row per:  
  **(geo, coicop, timeQuarter)**  
  (and optionally crossed with income dimensions if needed)
- Columns:
  - `hicpIndex` by COICOP (CP00–CP12)
  - `hicpInflation` by COICOP
  - Optional month-coverage flags (`nMonths`) for quality control
  - Optionally selected income and employment variables (if merged later)

Key idea:

- The **macro panel** keeps things simple and interpretable using only CP00.
- The **category panel** preserves the richer structure of inflation by consumption
  group (CP01–CP12), enabling deeper analysis of which components drive headline
  inflation or affect different income quantiles.

In the code that follows, I first construct the **macro panel** (CP00 only) and
save it, then build a second **category panel** that includes the `coicop`
dimension.

---

### 3. Sector panel (employment by NACE)

The third dataset is a **sector employment panel** based on NACE Rev.2 aggregates:

> How does **employment by sector** (industry, services, construction, etc.) evolve
> across countries and around major shocks (2008, 2012, 2020)?

Structure:

- One row per:  
  **(geo, nace_r2, timeQuarter)**
- Columns:
  - `employmentIndex` (same definition as in the macro panel:
    Q, I15_HW, SCA, EMP_DC, but split by NACE aggregate)

Typical NACE aggregates used:

- `"TOTAL"` – all NACE activities  
- `"A"` – Agriculture, forestry and fishing  
- `"B-E"` – Industry (except construction)  
- `"F"` – Construction  
- `"G-I"` – Trade, transport, accommodation and food services  
- `"J"` – Information and communication  
- `"K"` – Financial and insurance activities  
- `"L"` – Real estate activities  
- `"M_N"` – Professional, scientific, technical; administrative support  
- `"O-Q"` – Public administration, education, health  
- `"R-U"` – Arts, entertainment, other services

Key idea:

- This panel focuses on **labour-market dynamics and structural change**:
  sectoral job losses and recoveries around crises, and longer-term shifts
  between industry and services.
- It is built from the same employment source and filters as the macro panel,
  so results are comparable across datasets.

The sector panel is saved as, for example:

- `data_processed/employment_sector_panel.parquet`


In [10]:
# Slim HICP CP00 tables, keep their own nMonths
hicp_index_cp00 = hicp_index_cp00[["geo", "timeQuarter", "hicpIndex", "nMonths"]].copy()
hicp_index_cp00 = hicp_index_cp00.rename(columns={"nMonths": "hicpIndex_nMonths"})

hicp_inflation_cp00 = hicp_inflation_cp00[["geo", "timeQuarter", "hicpInflation", "nMonths"]].copy()
hicp_inflation_cp00 = hicp_inflation_cp00.rename(columns={"nMonths": "hicpInflation_nMonths"})

# Merge index + inflation
hicp_cp00 = pd.merge(
    hicp_index_cp00,
    hicp_inflation_cp00,
    on=["geo", "timeQuarter"],
    how="inner",
)

# Optional: a combined quality flag (e.g. min of both)
hicp_cp00["hicp_nMonths_min"] = hicp_cp00[["hicpIndex_nMonths", "hicpInflation_nMonths"]].min(axis=1)

print("HICP CP00 block:", hicp_cp00.shape)
hicp_cp00.head()


HICP CP00 block: (3600, 7)


,geo,timeQuarter,hicpIndex,hicpIndex_nMonths,hicpInflation,hicpInflation_nMonths,hicp_nMonths_min
0,AT,2000Q1,74.683333,3,1.733333,3,3
1,AT,2000Q2,74.83,3,1.933333,3,3
2,AT,2000Q3,74.923333,3,2.033333,3,3
3,AT,2000Q4,75.64,3,2.1,3,3
4,AT,2001Q1,76.166667,3,2.0,3,3


In [11]:
# Choose a single, interpretable income definition for the macro panel
income_q_macro = income_q_full.query('indic_il == "SHARE" and currency == "EUR"').copy()

income_base = income_q_macro[[
    "geo",
    "quantile",
    "indic_il",
    "currency",
    "timeQuarter",
    "incomeValue",
]].copy()


print("Income base:", income_base.shape)
income_base.head()


Income base: (108000, 6)


,geo,quantile,indic_il,currency,timeQuarter,incomeValue
0,AT,D1,SHARE,EUR,2000Q1,4.0
1,AT,D1,SHARE,EUR,2000Q2,4.0
2,AT,D1,SHARE,EUR,2000Q3,4.0
3,AT,D1,SHARE,EUR,2000Q4,4.0
4,AT,D1,SHARE,EUR,2001Q1,4.0


### Build the master Panel

In [12]:
# Build the master panel:
#   income (base)  +  HICP CP00 (with quality flags)  +  headline employment
panel = income_base.merge(
    hicp_cp00,
    on=["geo", "timeQuarter"],
    how="left",
).merge(
    emp_headline,
    on=["geo", "timeQuarter"],
    how="left",
)

print("Master panel shape:", panel.shape)
panel.head()



Master panel shape: (108000, 12)


,geo,quantile,indic_il,currency,timeQuarter,incomeValue,hicpIndex,hicpIndex_nMonths,hicpInflation,hicpInflation_nMonths,hicp_nMonths_min,employmentIndex
0,AT,D1,SHARE,EUR,2000Q1,4.0,74.683333,3,1.733333,3,3,98.0
1,AT,D1,SHARE,EUR,2000Q2,4.0,74.83,3,1.933333,3,3,98.3
2,AT,D1,SHARE,EUR,2000Q3,4.0,74.923333,3,2.033333,3,3,98.6
3,AT,D1,SHARE,EUR,2000Q4,4.0,75.64,3,2.1,3,3,99.5
4,AT,D1,SHARE,EUR,2001Q1,4.0,76.166667,3,2.0,3,3,99.2


In [13]:
print("Any missing HICP index?", panel["hicpIndex"].isna().sum())
print("Any missing HICP inflation?", panel["hicpInflation"].isna().sum())
print("Any missing employmentIndex?", panel["employmentIndex"].isna().sum())

# How many rows per geo-quarter (should be constant)
rows_per_gq = panel.groupby(["geo", "timeQuarter"]).size()
print(rows_per_gq.describe())
rows_per_gq.unique()[:10]


Any missing HICP index? 1740
Any missing HICP inflation? 2160
Any missing employmentIndex? 510
count    3600.0
mean       30.0
std         0.0
min        30.0
25%        30.0
50%        30.0
75%        30.0
max        30.0
dtype: float64


array([30])

In [14]:
# ... after you build `panel` and run the missing-values / rows_per_gq checks ...

# 1. Assert uniqueness of the full key
key_cols = ["geo", "quantile", "indic_il", "currency", "timeQuarter"]
assert not panel.duplicated(key_cols).any(), "Duplicate rows in master panel!"

# 2. Assert expected sizes
assert panel["geo"].nunique() == 36
assert panel["timeQuarter"].nunique() == 100
assert panel["quantile"].nunique() == 30

# 3. Fix column order explicitly
panel = panel[
    [
        "geo", "quantile", "indic_il", "currency", "timeQuarter",
        "incomeValue",
        "hicpIndex", "hicpIndex_nMonths",
        "hicpInflation", "hicpInflation_nMonths",
        "hicp_nMonths_min",
        "employmentIndex",
    ]
]


In [15]:
# Save macro master panel
master_panel_path = DATA_PROCESSED_DIR / "master_quarterly_panel.parquet"
panel.to_parquet(master_panel_path)

print("Saved master macro panel to:", master_panel_path)
print(panel.shape)


Saved master macro panel to: /Users/princebhanusteta/Documents/Projects/euro-macro-inflation-income/data_processed/master_quarterly_panel.parquet
(108000, 12)


###  Build Category panel (HICP by COICOP)

For the category panel I keep **exactly the same structure** as the macro
master panel (same rows, same income and employment variables), and I add
**12 extra features** with HICP *inflation by COICOP category* (CP01–CP12).

So each row is still:

> (geo, quantile, indic_il, currency, timeQuarter)

but now has additional columns:

> `hicpInflation_CP01` … `hicpInflation_CP12`

HICP and employment definitions (CP00 headline, NACE = TOTAL, hours worked)
stay identical to the macro panel, to keep everything consistent.

#### Why only category **inflation** (not category indices)?

- Category indices and category inflation carry almost the **same information**,
  but inflation is directly interpretable for economics questions
  (e.g. “food prices +8% y/y”).
- The macro panel already contains the **headline index and inflation** for CP00.
- Adding 12 more index series (CP01–CP12) would double the width of the panel
  without adding much extra signal for the kind of models and EDA I want to do.

So the COICOP panel is a **macro panel plus 12 interpretable inflation
features**, which keeps the dataset wide enough to be interesting but still
manageable for EDA and modelling.

This panel is saved as:

- `data_processed/master_quarterly_panel_coicop.parquet`


In [16]:
# HICP inflation categories: CP01–CP12 (exclude CP00, already in macro panel)
all_coicop_codes = sorted(hicp_inflation_q["coicop"].unique())
print("All COICOP codes in quarterly HICP inflation:")
print(all_coicop_codes)

coicop_cp01_12 = [f"CP{str(i).zfill(2)}" for i in range(1, 13)]
coicop_used = [c for c in coicop_cp01_12 if c in all_coicop_codes]

print("\nCOICOP codes used for category features (CP01–CP12):")
print(coicop_used)

# Long table: one row per (geo, timeQuarter, coicop) with inflation
cat_inf_long = (
    hicp_inflation_q[
        hicp_inflation_q["coicop"].isin(coicop_used)
    ][["geo", "timeQuarter", "coicop", "hicpInflation"]]
    .copy()
)

print("\nLong HICP category table shape:", cat_inf_long.shape)
cat_inf_long.head()



All COICOP codes in quarterly HICP inflation:
['CP00', 'CP01', 'CP02', 'CP03', 'CP04', 'CP05', 'CP06', 'CP07', 'CP08', 'CP09', 'CP10', 'CP11', 'CP12']

COICOP codes used for category features (CP01–CP12):
['CP01', 'CP02', 'CP03', 'CP04', 'CP05', 'CP06', 'CP07', 'CP08', 'CP09', 'CP10', 'CP11', 'CP12']

Long HICP category table shape: (43200, 4)


,geo,timeQuarter,coicop,hicpInflation
3600,AT,2000Q1,CP01,0.766667
3601,AT,2000Q2,CP01,1.366667
3602,AT,2000Q3,CP01,2.766667
3603,AT,2000Q4,CP01,1.6
3604,AT,2001Q1,CP01,2.133333


In [17]:
# Sanity check: one observation per geo–coicop–quarter
dup_counts = (
    cat_inf_long
    .groupby(["geo", "timeQuarter", "coicop"])["hicpInflation"]
    .size()
)

print("Max rows per (geo, timeQuarter, coicop):", dup_counts.max())
assert (dup_counts == 1).all(), "Duplicates in HICP category inflation!"

# Pivot to wide: columns hicpInflation_CP01 ... hicpInflation_CP12
cat_inf_wide = (
    cat_inf_long
    .pivot(index=["geo", "timeQuarter"], columns="coicop", values="hicpInflation")
)

# Rename columns to make meaning explicit
cat_inf_wide = cat_inf_wide.rename(
    columns=lambda c: f"hicpInflation_{c}"
).reset_index()

print("Wide HICP category feature table shape:", cat_inf_wide.shape)
cat_inf_wide.head()


Max rows per (geo, timeQuarter, coicop): 1
Wide HICP category feature table shape: (3600, 14)


coicop,geo,timeQuarter,hicpInflation_CP01,hicpInflation_CP02,hicpInflation_CP03,hicpInflation_CP04,hicpInflation_CP05,hicpInflation_CP06,hicpInflation_CP07,hicpInflation_CP08,hicpInflation_CP09,hicpInflation_CP10,hicpInflation_CP11,hicpInflation_CP12
0,AT,2000Q1,0.766667,1.3,0.666667,2.4,0.866667,2.0,5.666667,-4.666667,0.766667,3.533333,1.733333,0.8
1,AT,2000Q2,1.366667,0.4,-0.333333,2.566667,0.9,1.166667,6.433333,-4.066667,0.533333,4.2,1.9,1.033333
2,AT,2000Q3,2.766667,0.9,-2.2,3.766667,0.533333,1.466667,5.7,-6.166667,1.1,4.533333,2.033333,1.8
3,AT,2000Q4,1.6,1.233333,-0.966667,5.0,0.466667,4.1,5.233333,-4.166667,-0.333333,4.966667,2.866667,1.9
4,AT,2001Q1,2.133333,2.166667,-0.966667,4.366667,0.9,5.2,2.7,-2.533333,0.8,3.2,2.133333,2.566667


In [18]:
# Merge CP01–CP12 inflation features onto the macro master panel
panel_coicop = panel.merge(
    cat_inf_wide,
    on=["geo", "timeQuarter"],
    how="left",
)

print("Macro + COICOP panel shape:", panel_coicop.shape)
panel_coicop.head()

Macro + COICOP panel shape: (108000, 24)


,geo,quantile,indic_il,currency,timeQuarter,incomeValue,hicpIndex,hicpIndex_nMonths,hicpInflation,hicpInflation_nMonths,...,hicpInflation_CP03,hicpInflation_CP04,hicpInflation_CP05,hicpInflation_CP06,hicpInflation_CP07,hicpInflation_CP08,hicpInflation_CP09,hicpInflation_CP10,hicpInflation_CP11,hicpInflation_CP12
0,AT,D1,SHARE,EUR,2000Q1,4.0,74.683333,3,1.733333,3,...,0.666667,2.4,0.866667,2.0,5.666667,-4.666667,0.766667,3.533333,1.733333,0.8
1,AT,D1,SHARE,EUR,2000Q2,4.0,74.83,3,1.933333,3,...,-0.333333,2.566667,0.9,1.166667,6.433333,-4.066667,0.533333,4.2,1.9,1.033333
2,AT,D1,SHARE,EUR,2000Q3,4.0,74.923333,3,2.033333,3,...,-2.2,3.766667,0.533333,1.466667,5.7,-6.166667,1.1,4.533333,2.033333,1.8
3,AT,D1,SHARE,EUR,2000Q4,4.0,75.64,3,2.1,3,...,-0.966667,5.0,0.466667,4.1,5.233333,-4.166667,-0.333333,4.966667,2.866667,1.9
4,AT,D1,SHARE,EUR,2001Q1,4.0,76.166667,3,2.0,3,...,-0.966667,4.366667,0.9,5.2,2.7,-2.533333,0.8,3.2,2.133333,2.566667


In [19]:
# 1. No duplicates on the core key (same as macro panel)
key_cols = ["geo", "quantile", "indic_il", "currency", "timeQuarter"]
assert not panel_coicop.duplicated(key_cols).any(), "Duplicate rows after COICOP merge!"

# 2. Check that we still have the same number of rows as the macro panel
print("Original macro panel shape:", panel.shape)
print("Macro + COICOP panel shape:", panel_coicop.shape)

assert panel_coicop.shape[0] == panel.shape[0], "Row count changed after COICOP merge!"

# 3. Inspect missing values in the new category features
cat_cols = [c for c in panel_coicop.columns if c.startswith("hicpInflation_CP")]
print("\nMissing values per COICOP inflation column:")
print(panel_coicop[cat_cols].isna().sum())

Original macro panel shape: (108000, 12)
Macro + COICOP panel shape: (108000, 24)

Missing values per COICOP inflation column:
hicpInflation_CP01    2160
hicpInflation_CP02    2160
hicpInflation_CP03    2160
hicpInflation_CP04    2160
hicpInflation_CP05    2160
hicpInflation_CP06    2160
hicpInflation_CP07    2160
hicpInflation_CP08    2160
hicpInflation_CP09    2160
hicpInflation_CP10    2250
hicpInflation_CP11    2160
hicpInflation_CP12    2160
dtype: int64


In [20]:
# Save macro + COICOP category panel
coicop_panel_path = DATA_PROCESSED_DIR / "master_quarterly_panel_coicop.parquet"
panel_coicop.to_parquet(coicop_panel_path)

print("Saved macro + COICOP panel to:", coicop_panel_path)
print(panel_coicop.shape)

Saved macro + COICOP panel to: /Users/princebhanusteta/Documents/Projects/euro-macro-inflation-income/data_processed/master_quarterly_panel_coicop.parquet
(108000, 24)
